<a href="https://colab.research.google.com/github/EvenSol/NeqSim-Colab/blob/master/notebooks/process/capacity_and_bottleneck_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Capacity and Bottleneck Analysis

Demonstrate how a process can be screened for bottlenecks using utilization snapshots and simple equipment limits.


## Setup

Install imports and create a gas stream.


In [ ]:
# Install NeqSim when running in a fresh Colab session.
try:
    import neqsim
except ImportError:
    %pip install neqsim

import json
import pandas as pd
import matplotlib.pyplot as plt
from neqsim import jneqsim
from neqsim.thermo import TPflash

SystemSrkEos = jneqsim.thermo.system.SystemSrkEos
Stream = jneqsim.process.equipment.stream.Stream
ProcessSystem = jneqsim.process.processmodel.ProcessSystem

def make_gas(temperature_c=25.0, pressure_bara=60.0):
    fluid = SystemSrkEos(273.15 + temperature_c, pressure_bara)
    fluid.addComponent("nitrogen", 0.01)
    fluid.addComponent("CO2", 0.02)
    fluid.addComponent("methane", 0.86)
    fluid.addComponent("ethane", 0.07)
    fluid.addComponent("propane", 0.03)
    fluid.addComponent("n-butane", 0.01)
    fluid.setMixingRule("classic")
    TPflash(fluid)
    fluid.initProperties()
    return fluid


## Build a Compressor Case

Set a design power so the compressor has a meaningful utilization basis.


In [ ]:
Compressor = jneqsim.process.equipment.compressor.Compressor
fluid = make_gas(25.0, 45.0)
feed = Stream("Feed gas", fluid)
feed.setFlowRate(12000.0, "kg/hr")
compressor = Compressor("Export compressor", feed)
compressor.setOutletPressure(100.0)
compressor.getMechanicalDesign().setMaxDesignPower(2500.0)
process = ProcessSystem()
process.add(feed)
process.add(compressor)
process.run()


## Read the Utilization Snapshot

The snapshot summarizes unit constraints and identifies the current bottleneck.


In [ ]:
auto = process.getAutomation()
snapshot = json.loads(str(auto.getUtilizationSnapshot()))
print(json.dumps(snapshot.get("bottleneck"), indent=2))
pd.DataFrame(snapshot.get("units", []))


## Screen Operating Rate

Increase feed rate and track compressor utilization.


In [ ]:
rows = []
for flow in [6000, 9000, 12000, 15000, 18000]:
    feed.setFlowRate(float(flow), "kg/hr")
    process.run()
    snap = json.loads(str(process.getUtilizationSnapshotJson()))
    bottleneck = snap.get("bottleneck") or {}
    rows.append({"flow_kg_per_hr": flow, "bottleneck": bottleneck.get("name"), "utilization_pct": bottleneck.get("utilizationPercent")})
pd.DataFrame(rows)
